# 豊田中央研究所 論文一覧スクレイピング

今井先生の元タスク文書「Build a Researcher Database」の2番目の対象。
`https://www.tytlabs.co.jp/papers/` は `https://www.tytlabs.co.jp/ja/technology/papers.html` にリダイレクトされる。
確認の結果、ページネーションはなく、静的HTML1ページに全355件（2018〜2026年）が最初から埋め込まれている
（年タブはJSでの表示フィルタ切替のみで、サーバー再取得は発生しない）。

- 対象URL: `https://www.tytlabs.co.jp/ja/technology/papers.html`
- robots.txt確認済み、問題なし
- 構造: `<li entrydate="YYYY/MM/DD" class="... core_technology_jaElm material_jaElm ...">` の中に
  `div.item > a[href]`（リンク・日付・カテゴリアイコン・タイトル）と `ul.explanation`（著者=text1・掲載誌=text2・巻号ページ=text3）
- カテゴリはコア技術・材料・数理・電気電子・ヒューマンライフ・エネルギー環境・機械・カーボンニュートラル・電動化・量子・製造・循環経済・ウェルビーイング等の**複数ラベル**が同時に付く（1論文に複数タグ）。単一のカテゴリ列にはせず、タグの一覧として保持する。
- この一覧には「トヨタ所属著者の太字表記」はない（未来創成センターと違い、豊田中央研究所自身の論文一覧のため、掲載著者は基本的に同研究所所属という前提と思われる）。

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import re

import pandas as pd
import requests
from bs4 import BeautifulSoup

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'AGENTS.md').exists() and (candidate / 'data').is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError('リポジトリ直下を特定できません。リポジトリ内で実行してください。')

OUTPUT_DIR = ROOT / 'data' / 'derived' / 'toyota_official_scrape'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_URL = 'https://www.tytlabs.co.jp/ja/technology/papers.html'
USER_AGENT = 'Mozilla/5.0 (compatible; Sci-sci-research-bot/1.0; contact: r.ikura@msp-lab.org)'

RUN_AT_UTC = datetime.now(timezone.utc).isoformat()
print(f'RUN_AT_UTC = {RUN_AT_UTC}')

RUN_AT_UTC = 2026-07-27T02:59:52.343454+00:00


## 1. 取得

リダイレクト先を1回だけGETする（ページネーションがないため、他ページへのアクセスは発生しない）。

In [2]:
resp = requests.get(SOURCE_URL, headers={'User-Agent': USER_AGENT, 'Accept': 'text/html'}, timeout=30)
resp.raise_for_status()
print(f'HTTPステータス: {resp.status_code} (最終URL: {resp.url})')
print(f'取得サイズ: {len(resp.text)} 文字')

soup = BeautifulSoup(resp.text, 'html.parser')
items = soup.select('li[entrydate]')
print(f'論文アイテム数: {len(items)}')

HTTPステータス: 200 (最終URL: https://www.tytlabs.co.jp/ja/technology/papers.html)
取得サイズ: 406388 文字


論文アイテム数: 355


## 2. 構造化

各`<li>`から日付・カテゴリタグ・タイトル・著者・掲載誌・巻号ページ・リンクを抜き出す。

In [3]:
def parse_item(li):
    entrydate = li.get('entrydate')
    # カテゴリタグは class から no-img / normal / titleOnly を除いた残り(Elmサフィックスも除去)
    raw_classes = li.get('class', [])
    ignore = {'no-img', 'normal', 'titleOnly'}
    category_tags = [c.replace('_jaElm', '') for c in raw_classes if c not in ignore]

    div = li.select_one('div.item')
    a = div.select_one('a') if div else None
    link = a.get('href') if a else None

    title_el = div.select_one('.news_tx') if div else None
    title = title_el.get_text(strip=True) if title_el else None

    ul = div.select_one('ul.explanation') if div else None
    authors = ul.select_one('.text1').get_text(strip=True) if ul and ul.select_one('.text1') else None
    journal = ul.select_one('.text2').get_text(strip=True) if ul and ul.select_one('.text2') else None
    vol_pages = ul.select_one('.text3').get_text(strip=True) if ul and ul.select_one('.text3') else None

    return {
        'entrydate': entrydate,
        'year': int(entrydate.split('/')[0]) if entrydate else None,
        'category_tags': '; '.join(category_tags),
        'title': title,
        'authors_raw': authors,
        'journal': journal,
        'volume_pages': vol_pages,
        'link': link,
    }


rows = [parse_item(li) for li in items]
df = pd.DataFrame(rows)
print(f'行数: {len(df)}')
print(f'タイトルが空: {df["title"].isna().sum()}')
print(f'著者が空: {df["authors_raw"].isna().sum()}')
print(f'リンクが空: {df["link"].isna().sum()}')
df.head(5)

行数: 355
タイトルが空: 0
著者が空: 0
リンクが空: 8


,entrydate,year,category_tags,title,authors_raw,journal,volume_pages,link
0,2026/07/22,2026,core_technology; material,High-Solid-Content Kneading Degrades Carboxyme...,"Makino, S., Ishii, M., Kawauchi, S., Sasaki, M...",Journal of Colloid and Interface Science,"Vol. 708, (2025), 139769.",https://doi.org/10.1016/j.jcis.2025.139769
1,2026/07/13,2026,core_technology; material,Electrostatic Dry Coating Strategies for Throu...,"Yonaga, A., Matsunaga, T.",Small Methods,"Vol. 10, No. 9, (2026), e02213.",https://doi.org/10.1002/smtd.202502213
2,2026/07/08,2026,core_technology; material; energy_environment,Photocatalytic CO2 reduction with a semiconduc...,"Morikawa, T., Suzuki, T. M., Yamaguchi, Y., Ku...",Journal of Photochemistry & Photobiology C: Ph...,"Vol. 66, (2026), 100735.",https://doi.org/10.1016/j.jphotochemrev.2025.1...
3,2026/06/19,2026,core_technology; electricity_electronics,Electrically Reconfigurable Topological RF Fil...,"Ozaki, T., Ohta, N., Ma, J., Fujiyoshi, M.",Communications Physics,"Vol. 9, (2026), 208.",https://doi.org/10.1038/s42005-026-02621-8
4,2026/06/18,2026,core_technology; material,Water Generation in the Mesoporous Catalyst La...,"Harada, M., Hasegawa, N., Kato, A., Higuchi, Y...",Chemical Engineering Journal,"Vol. 525, (2025), 170753.",https://doi.org/10.1016/j.cej.2025.170753


## 3. 保存

In [4]:
raw_html_path = OUTPUT_DIR / 'toyota_central_rd_labs_raw.html'
raw_html_path.write_text(resp.text, encoding='utf-8')

csv_path = OUTPUT_DIR / 'toyota_central_rd_labs.csv'
df.to_csv(csv_path, index=False)

meta_path = OUTPUT_DIR / 'toyota_central_rd_labs_meta.json'
with meta_path.open('w', encoding='utf-8') as f:
    json.dump({'run_at_utc': RUN_AT_UTC, 'source_url': SOURCE_URL, 'n_items': len(df)}, f, ensure_ascii=False, indent=2)

print(f'保存: {raw_html_path.relative_to(ROOT)}')
print(f'保存: {csv_path.relative_to(ROOT)}')
print(f'保存: {meta_path.relative_to(ROOT)}')

保存: data/derived/toyota_official_scrape/toyota_central_rd_labs_raw.html
保存: data/derived/toyota_official_scrape/toyota_central_rd_labs.csv
保存: data/derived/toyota_official_scrape/toyota_central_rd_labs_meta.json


## 4. ざっと確認

In [5]:
print('年別件数:')
print(df['year'].value_counts().sort_index())
print()
print('カテゴリタグ別件数(1論文が複数タグを持つため延べ数):')
from collections import Counter
tag_counter = Counter()
for tags in df['category_tags']:
    for t in tags.split('; '):
        if t:
            tag_counter[t] += 1
for tag, n in tag_counter.most_common():
    print(f'  {tag}: {n}')

年別件数:
year
2018    18
2019    54
2020    64
2021    48
2022    27
2023    45
2024    39
2025    32
2026    28
Name: count, dtype: int64

カテゴリタグ別件数(1論文が複数タグを持つため延べ数):
  core_technology: 354
  material: 147
  mathematics: 105
  project: 68
  energy_environment: 50
  electricity_electronics: 43
  human_life: 34
  machine: 34
  carbonneutral: 26
  electrification: 10
  quantum: 9
  well-being: 9
  circulareconomy: 8
  manufacturing: 7
